<a href="https://colab.research.google.com/github/09-ashish/DSA-CaseStudies/blob/main/Predicting_the_Sex_of_Laysan_Albatross.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

In [4]:
df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [12]:
df.isnull().sum()
df.dropna(inplace=True)

In [14]:
target_mapping = {'H': 0, 'M': 1}
df['sexo_encoded'] = df['sexo'].map(target_mapping)

X = df.drop(columns=['id', 'sexo', 'sexo_encoded'], errors='ignore')
y = df['sexo_encoded']

X_test = test_df.drop(columns=['id'], errors='ignore')

print("Target value counts:")
print(y.value_counts())
print("\nFeature matrix shape:", X.shape)
print("Test feature matrix shape:", X_test.shape)

Target value counts:
sexo_encoded
0    57
1    36
Name: count, dtype: int64

Feature matrix shape: (93, 12)
Test feature matrix shape: (41, 12)


In [15]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

feature_names = X.columns.tolist()

imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

X_imputed = imputer.fit_transform(X)
X_test_imputed = imputer.transform(X_test)

X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=feature_names)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imputed), columns=feature_names)

print("Scaled Training Features Shape:", X_scaled.shape)
print("Scaled Test Features Shape:", X_test_scaled.shape)
X_scaled.head()

Scaled Training Features Shape: (93, 12)
Scaled Test Features Shape: (41, 12)


,longitudCraneo,longitudPico,longitudNarina,anchoCraneo,altoPico,anchoPico,tarso,longAlaCerrada,longAlaAbierta,mediaEnvergadura,envergadura,peso
0,-0.430874,-0.810458,-0.855452,-0.121999,-1.685525,-1.542247,-0.748932,-0.557236,-0.259444,-0.225887,-0.225887,-0.778058
1,-0.998366,-1.383885,-1.089025,-0.702342,-1.387886,-0.992846,-0.229098,-0.979935,-0.854075,-1.030310,-1.030310,-1.541745
2,-0.336292,-0.815533,-0.239954,-1.624592,-1.395146,0.718751,0.192388,0.879942,0.152224,0.176324,0.176324,-0.396214
3,1.065748,0.793106,0.388169,-0.198478,-0.320744,0.437007,0.304785,0.795402,1.204264,1.141632,1.141632,0.653856
4,-0.519892,-1.231647,-0.798637,-1.575105,1.073075,1.176585,-1.089633,-0.726316,-0.808335,-0.426993,-0.426993,0.335653


In [16]:

lgb_model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    random_state=42,
    verbose=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(lgb_model, X_scaled, y, cv=cv, scoring='f1_macro')

print(f"5-Fold Cross-Validation Macro F1-Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

lgb_model.fit(X_scaled, y)

predictions = lgb_model.predict(X_test_scaled)

test_ids = test_df['id'] if 'id' in test_df.columns else test_df.index
submission = pd.DataFrame({
    'id': test_ids,
    'sexo': predictions
})



submission.to_csv('submission.csv', index=False)
print("Saved submission.csv successfully!")

5-Fold Cross-Validation Macro F1-Score: 0.8408 (+/- 0.1443)
Saved submission.csv successfully!


In [17]:
# 1. Map binary predictions back to string labels
inverse_target_mapping = {0: 'H', 1: 'M'}
submission['sexo'] = submission['sexo'].map(inverse_target_mapping)

# 2. Re-save submission
submission.to_csv('submission.csv', index=False)

# 3. Verify submission format
print(submission.head())

         id sexo
0  A70_2018    M
1  5C9_2018    H
2  5E1_2018    M
3  C58_2018    M
4  5E2_2018    M


In [18]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model_params = {
    'LightGBM': {
        'model': LGBMClassifier(random_state=42, verbose=-1),
        'params': {
            'n_estimators': [50, 100, 200],
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'max_depth': [3, 5, -1],
            'num_leaves': [15, 31, 63]
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200, 300],
            'max_depth': [3, 5, 8, None],
            'min_samples_split': [2, 5, 10]
        }
    },
    'XGBoost': {
        'model': XGBClassifier(random_state=42, eval_metric='logloss'),
        'params': {
            'n_estimators': [50, 100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7]
        }
    }
}

best_models = {}
results = []

for name, mp in model_params.items():
    clf = GridSearchCV(mp['model'], mp['params'], cv=cv, scoring='f1_macro', n_jobs=-1)
    clf.fit(X_scaled, y)
    best_models[name] = clf.best_estimator_
    results.append({'Model': name, 'Best Macro F1': round(clf.best_score_, 4), 'Params': clf.best_params_})


results_df = pd.DataFrame(results).sort_values(by='Best Macro F1', ascending=False)
print(results_df[['Model', 'Best Macro F1']])

           Model  Best Macro F1
2        XGBoost         0.8679
0       LightGBM         0.8494
1  Random Forest         0.8491


In [19]:

best_model = best_models['XGBoost']


best_model.fit(X_scaled, y)


predictions = best_model.predict(X_test_scaled)

inverse_target_mapping = {0: 'H', 1: 'M'}
submission['sexo'] = pd.Series(predictions).map(inverse_target_mapping)

submission.to_csv('submission.csv', index=False)

print(submission.head())

         id sexo
0  A70_2018    M
1  5C9_2018    H
2  5E1_2018    H
3  C58_2018    M
4  5E2_2018    M
